# Free long-form AI video on a Kaggle T4 (Wan 2.2 TI2V-5B)

Renders chained scenes from `projects/example/storyboard.json` using this repo's pipeline.

**Read `docs/KAGGLE_COLAB_GPU.md` first.** Key facts:
- Kaggle gives ~30 GPU-hours/week on 2x T4 (16 GB each).
- Wan 2.2 **TI2V-5B** (Apache-2.0) fits comfortably; the 14B FLF2V needs 16-24 GB quantised.
- Sessions die. That is expected: the pipeline keeps SQLite state, so `resume` re-renders
  only the unfinished scenes. Save `projects/<slug>/` between sessions.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch, sys
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("python", sys.version.split()[0])

In [ ]:
# 1. Get ComfyUI (the engine) and this pipeline (the orchestrator)
%cd /kaggle/working
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI
!git clone --depth 1 https://github.com/omareg03/Agent-Best---Ai-for-Automation pipeline
%cd /kaggle/working/pipeline
!pip install -q -r requirements.txt
import os; os.environ["PYTHONPATH"] = "/kaggle/working/pipeline"
print("installed")

In [ ]:
# 2. ComfyUI dependencies
%cd /kaggle/working/ComfyUI
!pip install -q -r requirements.txt
print("comfy ready")

In [ ]:
# 3. Weights (~18 GB). TI2V-5B = text+image to video, 720p, 24 fps, Apache-2.0.
%cd /kaggle/working/ComfyUI
!mkdir -p models/diffusion_models models/text_encoders models/vae
BASE=https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files
!wget -q -c -P models/diffusion_models $BASE/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors
!wget -q -c -P models/text_encoders    $BASE/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
!wget -q -c -P models/vae              $BASE/vae/wan2.2_vae.safetensors
!du -sh models/*/*

In [ ]:
# 4. Start ComfyUI (keeps running in the background)
%cd /kaggle/working/ComfyUI
import subprocess, time, urllib.request
proc = subprocess.Popen(["python","main.py","--listen","0.0.0.0","--port","8188"],
                        stdout=open("/kaggle/working/comfy.log","w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/object_info", timeout=2); break
    except Exception:
        time.sleep(2)
print("ComfyUI up on :8188 (log: /kaggle/working/comfy.log)")

In [ ]:
# 5. PROVE the node classes exist before spending GPU hours.
# This diffs every class_type in our graph against the server's /object_info.
import os, subprocess
os.environ["PYTHONPATH"] = "/kaggle/working/pipeline"
os.environ["COMFYUI_URL"] = "http://127.0.0.1:8188"
print(subprocess.run(["python","/kaggle/working/pipeline/scripts/validate_comfy.py",
                      "--graph","wan22_ti2v_5b"], capture_output=True, text=True).stdout)
print("If node classes are missing above, STOP: install the matching nodes first.")

In [ ]:
# 6. Smoke test: ONE scene, then inspect the seam before rendering the rest.
import os, subprocess
os.environ["PYTHONPATH"] = "/kaggle/working/pipeline"
cmd = ["python","-m","scripts.video_pipeline.cli","run","example",
       "--engine","wan22_ti2v_5b","--limit","1","--json"]
r = subprocess.run(cmd, cwd="/kaggle/working/pipeline", capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-2000:])

## Interruptions

If the session dies mid-render, on the next session restore `projects/example/`
(save it as a Kaggle dataset, or to Drive) and run `resume`. Only scenes whose
artefacts are missing — or whose prompt hash changed — are re-rendered.

```bash
python -m scripts.video_pipeline.cli status example
python -m scripts.video_pipeline.cli resume example --engine wan22_ti2v_5b
```

Do not delete `state.db`: it is the memory of what has already been paid for.